# Distributed Training for Large Models

**Estimated time: 20 minutes** (conceptual overview)

**Note:** This notebook requires a multi-node or multi-GPU setup. It provides conceptual understanding and code templates.

## Learning Objectives
- Understand distributed training strategies
- Learn about Data Parallel vs Model Parallel
- Configure PyTorch DDP (DistributedDataParallel)
- Optimize communication overhead

## Introduction

When training very large models (13B+ parameters), single-GPU training may not be feasible. Distributed training allows us to split the workload across multiple GPUs or nodes.

## Part 1: Distributed Training Strategies

### Data Parallel (DP)
- Each GPU gets a copy of the model
- Data is split across GPUs
- Gradients are averaged across GPUs
- **Use when:** Model fits on one GPU

### Model Parallel (MP)
- Model is split across GPUs
- Each GPU holds part of the model
- **Use when:** Model too large for one GPU

### Pipeline Parallel (PP)
- Model layers split into stages
- Micro-batches flow through pipeline
- **Use when:** Very deep models

### Tensor Parallel (TP)
- Individual layers split across GPUs
- Matrix multiplications distributed
- **Use when:** Wide layers, fast interconnect

### Hybrid Strategies
- Combine multiple approaches
- Example: DP + TP for Llama-70B

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize different parallelism strategies
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Data Parallel
ax = axes[0, 0]
for i in range(4):
    ax.add_patch(plt.Rectangle((i*2, 0), 1.5, 3, fill=True, color=f'C{i}', alpha=0.7))
    ax.text(i*2 + 0.75, 1.5, f'GPU {i}\nFull Model', ha='center', va='center', fontweight='bold')
ax.set_xlim(-0.5, 8)
ax.set_ylim(-0.5, 3.5)
ax.set_title('Data Parallel\n(Each GPU has full model, different data)', fontsize=12, fontweight='bold')
ax.axis('off')

# Model Parallel
ax = axes[0, 1]
colors = ['C0', 'C1', 'C2', 'C3']
for i, color in enumerate(colors):
    ax.add_patch(plt.Rectangle((0, i*0.7), 6, 0.6, fill=True, color=color, alpha=0.7))
    ax.text(3, i*0.7 + 0.3, f'GPU {i}: Layers {i*8}-{(i+1)*8-1}', ha='center', va='center')
ax.set_xlim(-0.5, 6.5)
ax.set_ylim(-0.5, 3.5)
ax.set_title('Model Parallel\n(Model split across GPUs)', fontsize=12, fontweight='bold')
ax.axis('off')

# Pipeline Parallel
ax = axes[1, 0]
for i in range(4):
    for j in range(3):
        alpha = 0.3 + 0.2 * j
        ax.add_patch(plt.Rectangle((i*2 + j*0.3, j), 1.2, 0.8, fill=True, color=f'C{i}', alpha=alpha))
    ax.text(i*2 + 0.6, -0.5, f'GPU {i}\nStage {i}', ha='center')
ax.arrow(1, 1.5, 5, 0, head_width=0.2, head_length=0.3, fc='black', ec='black')
ax.text(3.5, 2, 'Pipeline Flow →', ha='center')
ax.set_xlim(-0.5, 8)
ax.set_ylim(-1, 4)
ax.set_title('Pipeline Parallel\n(Stages flow through pipeline)', fontsize=12, fontweight='bold')
ax.axis('off')

# Tensor Parallel
ax = axes[1, 1]
for i in range(4):
    for j in range(4):
        ax.add_patch(plt.Rectangle((i*1.5, j*0.7), 1.2, 0.6, fill=True, color=f'C{i}', alpha=0.7))
ax.text(3, -0.5, 'Each layer split across GPUs', ha='center')
for i in range(4):
    ax.text(i*1.5 + 0.6, -1, f'GPU {i}', ha='center')
ax.set_xlim(-0.5, 7)
ax.set_ylim(-1.5, 3.5)
ax.set_title('Tensor Parallel\n(Each layer split column-wise)', fontsize=12, fontweight='bold')
ax.axis('off')

plt.suptitle('Distributed Training Strategies', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## Part 2: PyTorch DistributedDataParallel (DDP)

DDP is the recommended approach for data-parallel training in PyTorch.

In [ ]:
# Template for DDP training script

ddp_template = '''
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def setup(rank, world_size):
    """Initialize the distributed environment"""
    dist.init_process_group(
        backend="nccl",  # Use NCCL for GPU training
        init_method="tcp://localhost:12355",
        world_size=world_size,
        rank=rank
    )
    torch.cuda.set_device(rank)

def cleanup():
    """Clean up the distributed environment"""
    dist.destroy_process_group()

def train(rank, world_size, model, dataset, epochs=3):
    """Training function for each process"""
    setup(rank, world_size)
    
    # Wrap model with DDP
    model = model.to(rank)
    ddp_model = DDP(model, device_ids=[rank])
    
    # Create DistributedSampler
    sampler = DistributedSampler(
        dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=32,
        sampler=sampler,
        num_workers=4,
        pin_memory=True
    )
    
    optimizer = torch.optim.AdamW(ddp_model.parameters(), lr=2e-4)
    
    # Training loop
    for epoch in range(epochs):
        sampler.set_epoch(epoch)  # Important for proper shuffling
        
        for batch in dataloader:
            optimizer.zero_grad()
            
            inputs = batch["input_ids"].to(rank)
            labels = batch["labels"].to(rank)
            
            outputs = ddp_model(input_ids=inputs, labels=labels)
            loss = outputs.loss
            
            loss.backward()
            optimizer.step()
            
            if rank == 0:  # Only log from main process
                print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
    
    cleanup()

def main():
    world_size = torch.cuda.device_count()
    print(f"Training on {world_size} GPUs")
    
    mp.spawn(
        train,
        args=(world_size, model, dataset),
        nprocs=world_size,
        join=True
    )

if __name__ == "__main__":
    main()
'''

print("DDP Training Template:")
print(ddp_template)

## Part 3: Launch Scripts

### Single Node, Multiple GPUs

```bash
# Using torchrun (recommended)
torchrun --nproc_per_node=4 train_ddp.py

# Using torch.distributed.launch (older)
python -m torch.distributed.launch \
    --nproc_per_node=4 \
    train_ddp.py
```

### Multiple Nodes

```bash
# On Node 0 (master)
torchrun \
    --nproc_per_node=4 \
    --nnodes=2 \
    --node_rank=0 \
    --master_addr="192.168.1.1" \
    --master_port=12355 \
    train_ddp.py

# On Node 1
torchrun \
    --nproc_per_node=4 \
    --nnodes=2 \
    --node_rank=1 \
    --master_addr="192.168.1.1" \
    --master_port=12355 \
    train_ddp.py
```

## Part 4: DeepSpeed Integration

DeepSpeed provides advanced optimizations for distributed training.

In [ ]:
# DeepSpeed configuration example
deepspeed_config = {
    "train_batch_size": 32,
    "gradient_accumulation_steps": 4,
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 2e-4,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
            "weight_decay": 0.01
        }
    },
    "fp16": {
        "enabled": True,
        "loss_scale": 0,
        "initial_scale_power": 16,
        "loss_scale_window": 1000
    },
    "zero_optimization": {
        "stage": 2,  # ZeRO Stage 2: Optimizer + Gradient Partitioning
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "contiguous_gradients": True,
        "overlap_comm": True
    },
    "activation_checkpointing": {
        "partition_activations": True,
        "contiguous_memory_optimization": True
    }
}

import json
print("DeepSpeed Configuration:")
print(json.dumps(deepspeed_config, indent=2))

### ZeRO Stages Explained

| Stage | Partitioned | Memory Savings | Communication Overhead |
|-------|------------|----------------|------------------------|
| **Stage 0** | Nothing | None | None |
| **Stage 1** | Optimizer States | 4x | Low |
| **Stage 2** | Optimizer + Gradients | 8x | Medium |
| **Stage 3** | Optimizer + Gradients + Weights | Linear w/ GPUs | High |

**Recommendation:**
- Use Stage 2 for most cases
- Use Stage 3 only when model doesn't fit otherwise

## Part 5: Performance Optimization

### Tips for Efficient Distributed Training

1. **Gradient Accumulation**
   - Simulates larger batch sizes
   - Reduces communication frequency
   
2. **Mixed Precision Training**
   - Use FP16 or BF16
   - Reduces memory and speeds up computation
   
3. **Gradient Checkpointing**
   - Trade computation for memory
   - Enables larger models
   
4. **Efficient Communication**
   - Use NCCL backend for GPUs
   - Ensure fast interconnect (NVLink, InfiniBand)
   
5. **Data Loading**
   - Use multiple workers
   - Pin memory
   - Prefetch data

In [ ]:
# Performance comparison visualization
gpus = [1, 2, 4, 8]
ideal_speedup = gpus
actual_ddp = [1, 1.85, 3.5, 6.8]  # Typical DDP speedup
actual_deepspeed = [1, 1.9, 3.7, 7.2]  # DeepSpeed with optimizations

plt.figure(figsize=(10, 6))
plt.plot(gpus, ideal_speedup, 'k--', label='Ideal (Linear)', linewidth=2)
plt.plot(gpus, actual_ddp, 'o-', label='PyTorch DDP', linewidth=2, markersize=8)
plt.plot(gpus, actual_deepspeed, 's-', label='DeepSpeed ZeRO-2', linewidth=2, markersize=8)

plt.xlabel('Number of GPUs', fontsize=12)
plt.ylabel('Speedup', fontsize=12)
plt.title('Distributed Training Speedup Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(gpus)
plt.tight_layout()
plt.show()

# Calculate efficiency
print("\nScaling Efficiency:")
print(f"{'GPUs':>5} | {'DDP Efficiency':>15} | {'DeepSpeed Efficiency':>20}")
print("-" * 50)
for i, n in enumerate(gpus):
    ddp_eff = (actual_ddp[i] / n) * 100
    ds_eff = (actual_deepspeed[i] / n) * 100
    print(f"{n:>5} | {ddp_eff:>14.1f}% | {ds_eff:>19.1f}%")

## Summary

### Key Takeaways:

1. **Parallelism Strategies**:
   - **Data Parallel**: Model fits on one GPU, split data
   - **Model Parallel**: Model too large, split model
   - **Pipeline Parallel**: Very deep models
   - **Tensor Parallel**: Wide layers

2. **PyTorch DDP**:
   - Recommended for data-parallel training
   - Use DistributedSampler
   - Launch with torchrun

3. **DeepSpeed ZeRO**:
   - Stage 2 for most cases
   - Stage 3 for very large models
   - Significant memory savings

4. **Optimization**:
   - Gradient accumulation
   - Mixed precision
   - Gradient checkpointing
   - Efficient data loading

### Recommended Setup for Llama-3.1-8B:
- **Single Node (4x GPUs)**: DDP with gradient accumulation
- **Llama-13B**: DeepSpeed ZeRO-2
- **Llama-70B**: DeepSpeed ZeRO-3 + Tensor Parallel

### References:
- [PyTorch DDP Tutorial](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)
- [DeepSpeed Documentation](https://www.deepspeed.ai/)
- [Megatron-LM](https://github.com/NVIDIA/Megatron-LM)